In [92]:
import pandas as pd
from IPython.display import display

df = pd.read_csv('data.csv')
num_cols = df.select_dtypes(include='number').columns
cat_cols = df.select_dtypes(exclude='number').columns

In [93]:
print('Rozmiar zbioru:', df.shape)
print("WNIOSKI: 418 wierszy to niewiele")

Rozmiar zbioru: (418, 20)
WNIOSKI: 418 wierszy to niewiele


In [94]:
print('\nTypy danych:')
display(df.dtypes.to_frame('dtype'))


Typy danych:


,dtype
ID,int64
N_Days,int64
Status,str
Drug,str
Age,int64
Sex,str
Ascites,str
Hepatomegaly,str
Spiders,str
Edema,str


In [ ]:
missing_info = df.isna().sum().to_frame('missing_values')
missing_info['percentage'] = (df.isna().mean() * 100).round(2).astype(str) + '%'

print('\nBraki danych w kolumnach:')
display(missing_info)

print("WNIOSKI: 9 kolumn ma braki danych powyżej 25%")

In [ ]:
print('\nPodstawowe statystyki dla cech liczbowych:')
display(df[num_cols].describe().T)

print("WNIOSKI:")
print("najmłodszy pacjent miał 26 lat, natomiast wiekszość pacjentów (25%-75%) jest w wieku pomiędzy 42-58 lat")
print("czas obserwacji (n-days) jest bardzo rozpięty, sugeruje to duże zróżnicowanie postępowania choroby/bardzo różne fazy rozpoznania choroby i udania się do szpitala")

In [ ]:
print(df['Stage'].value_counts().sort_index())
print("Widzimy że zdecydowana wiekszość pacjentów w zbiorze jest w ciężkim stanie (3 i 4 stadium choroby)")

In [ ]:
for col in cat_cols:
    print(f"\nCecha: {col}")
    counts = df[col].value_counts(dropna=False)
    percents = df[col].value_counts(normalize=True, dropna=False) * 100
    summary = pd.concat([counts, percents.round(2)], axis=1, keys=['Liczba', 'Procent %'])
    display(summary)

print("WNIOSKI")
print("zdecydowana wiekszość wierszy to kobiety")
print("liczność ludzi przyjmujących placebo i d-pecicylamine jest prawie równe")
print("bardzo mało osób miało przeszczep, prawie 40% przypadków ze zbioru zakonczyło się śmiercią")


In [ ]:
print('\nRozkład etykiety Stage:')
display(df['Stage'].value_counts(dropna=False).sort_index().to_frame('count'))
print("widzimy że zdecydowana wiekszość przypadków w zbiorze to przypadki o dwóch najwyższych stadiach choroby, co może tłumaczyć wysoką śmiertelność, która była wnioskiem w poprzedniej komórce")


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 8))
corr_matrix = df[num_cols].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", square=True)
plt.title('Macierz korelacji dla cech liczbowych')
plt.show()

print("WNIOSKI:")
print("1. Występuje zauważalna dodatnia korelacja (0.43) między poziomem albumin a liczbą dni (N_Days). Może to oznaczać, że pacjenci z wyższym poziomem tego białka charakteryzują się dłuższym okresem przeżycia (lub dłuższą obserwacją w badaniu).")
print("2. Czas (N_Days) jest ujemnie skorelowany z poziomem bilirubiny (-0.40), stadium choroby (Stage: -0.37) oraz poziomem miedzi (-0.36). Im bardziej zaawansowane stadium i wyższe stężenie tych toksycznych substancji, tym krótszy czas obserwacji pacjenta, co najprawdopodobniej wiąże się z szybszym zgonem.")
print("3. Zmienne takie jak Wiek (Age) oraz Płytki krwi (Platelets) są bardzo słabo skorelowane z większością pozostałych cech. Oznacza to, że wiek pacjenta nie determinuje wprost wartości tych wyników biochemicznych w tym konkretnym zbiorze danych.")


In [ ]:
crosstab_drug = pd.crosstab(df['Drug'], df['Status'], normalize='index') * 100
display(crosstab_drug.round(2).astype(str) + '%')

drug_stats = df.groupby('Drug')[['Bilirubin', 'Prothrombin', 'Copper']].mean()
display(drug_stats.round(2))

print("WNIOSKI:")
print("1. Branie leku vs placebo nie wpływa znacząco na przeżywalność pacjenta")
print("2. Średni poziom bilirubiny jest niższy u pacjentów leczonych D-penicylaminą (2.87) w porównaniu do grupy placebo (3.65), co może sugerować, że lek ma łagodzący wpływ na ten konkretny parametr wątrobowy.")



In [ ]:
df['Age_years'] = df['Age'] / 365.25

plt.figure(figsize=(10, 6))
sns.histplot(data=df, x='Age_years', hue='Status', multiple='stack', bins=20, palette='Set1')
plt.title('Dystrybucja wieku pacjentów (w latach) w podziale na Status')
plt.xlabel('Wiek (lata)')
plt.ylabel('Liczba pacjentów')
plt.show()

print("WNIOSKI:")
print("- Rozkład zgonów (Status D) jest zauważalnie przesunięty w prawo, co oznacza, że starsi pacjenci (szczególnie powyżej 45 roku życia) stanowią największą grupę ryzyka.")
print("- W grupie pacjentów nie seniorow (przed 60), znacznie częściej obserwujemy status C (pacjenci ocalejący/żyjący na koniec badań).")